<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Loss and gradients

**[Loss and gradients](https://learning.nextia-ai.com/courses/math/m04/loss-and-gradients/)** · Essential Mathematics and Statistics for AI · Module 4, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** calculate the loss (the mean squared error) of the ticket model for any `w`, find the gradient as the slope of the loss, make steps of gradient descent with the update rule, and measure two partial derivatives.

| | |
|---|---|
| **Time** | About 25 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | The ticket model minutes = w × length, the slope with units and the derivative as the slope of a tangent, from [Functions and change: The slope at one point](https://learning.nextia-ai.com/courses/math/m04/functions-and-change/#the-slope-at-one-point-the-derivative). Squared distances and their mean, from [Describe a dataset: Spread](https://learning.nextia-ai.com/courses/math/m01/describe-a-dataset/#spread-range-variance-and-standard-deviation). |
| **Tested** | 2026-10-07, Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m04/loss-and-gradients/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M04-L02-loss-and-gradients/loss-and-gradients-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. How wrong is w = 6? The loss

Tomás's model is minutes = w × length. For each ticket, the **error** is the actual minutes minus the predicted minutes. Priya squares each error and takes the mean of the squares: the **mean squared error** (MSE). A **loss** is one number that measures how wrong a model is. Here the loss is the MSE.

> **Predict.** At `w = 6`, the errors are 1, 1, −1 and 2. What is the sum of the squared errors? Then run the cell.

In [ ]:
tickets = ["T-3101", "T-3102", "T-3103", "T-3104"]
lengths = [1, 2, 3, 4]       # hundreds of words
minutes = [7, 13, 17, 26]    # actual handling minutes

w = 6
rows = []
for t, x, y in zip(tickets, lengths, minutes):
    prediction = w * x
    error = y - prediction
    rows.append([t, x, y, prediction, error, error ** 2])
table(["Ticket", "Length x", "Actual y", "Prediction", "Error", "Squared error"], rows)
total = sum(r[5] for r in rows)
print()
print("sum of squared errors:", total, "| loss (MSE):", total / len(rows))

> **Check.** You should see a sum of 7 and a loss of 1.75 squared minutes. T-3104 is wrong by 2 minutes but adds 4 to the sum: the square makes large errors count more. It also removes the sign, so errors of +1 and −1 cannot cancel.

## 2. The loss curve

The loss is a function of `w`. The next cell writes it as a Python function and prints Priya's table for `w` = 2, 5, 6 and 7.

Run it. You should see the losses 137.75, 13.25, 1.75 and 5.25.

In [ ]:
def loss(w):
    """Mean squared error of the model minutes = w × length on the four tickets."""
    squared = [(y - w * x) ** 2 for x, y in zip(lengths, minutes)]
    return sum(squared) / len(squared)

table(["w", "Loss (MSE)"], [[w, loss(w)] for w in [2, 5, 6, 7]])

The lesson draws the loss for many values of `w`. The next cell draws the same curve as text bars, turned on its side: one bar for each `w` from 4 to 8.5.

> **Predict.** Where is the shortest bar? Then run the cell.

In [ ]:
ws = [4 + i * 0.25 for i in range(19)]
bars([f"w = {w:.2f}" for w in ws], [round(loss(w), 2) for w in ws])

> **Check.** You should see a bowl on its side. The shortest bar is at `w = 6.25`, with a loss of 1.22. The loss never reaches 0, because no straight line goes through all four tickets.

The next cell tries every `w` from 2 to 10.5, in steps of 0.01, and keeps the one with the lowest loss. Run it. You should see a best `w` of 6.27 and a lowest loss of 1.217.

In [ ]:
candidates = [i / 100 for i in range(200, 1051)]
best_w = min(candidates, key=loss)
print("best w:", best_w, "| lowest loss:", round(loss(best_w), 3))

This search works for one parameter. A real model has millions of parameters, so nobody can try every value. Training only knows the slope where it stands.

## 3. The gradient: the slope of the loss

The **gradient** is the slope of the loss at the current `w`: the derivative of the loss with respect to the parameter. Its sign gives the direction, and its size gives the steepness. In the last lesson, you measured a slope with a very small gap. Do the same with the loss.

For this model, the gradient also has a short formula: 15w − 94. The next cell compares both.

> **Predict.** At `w = 2`, is the gradient positive or negative? Then run the cell.

In [ ]:
def slope_at(f, x, gap=0.000001):
    """The slope of f at x, measured with a very small gap (from the last lesson)."""
    return (f(x + gap) - f(x)) / gap

def gradient(w):
    """The gradient of the loss of the ticket model: 15w − 94."""
    return 15 * w - 94

table(["w", "Slope with a small gap", "Formula 15w − 94"],
      [[w, round(slope_at(loss, w), 3), gradient(w)] for w in [2, 5, 6, 7]], digits=3)

> **Check.** You should see −64, −19, −4 and 11 in both columns. A negative gradient means the loss falls to the right, so a larger `w` is better. At `w = 7`, the gradient is positive, so a smaller `w` is better.

### Where the gradient comes from

The gradient is a mean over the tickets. Each ticket adds its own part: 2 × (prediction − actual) × length. Run the cell. You should see the parts −2, −4, 6 and −16, and their mean, −4.

In [ ]:
w = 6
parts = [2 * (w * x - y) * x for x, y in zip(lengths, minutes)]
table(["Ticket", "Prediction − actual", "This ticket's part"],
      [[t, w * x - y, p] for t, x, y, p in zip(tickets, lengths, minutes, parts)])
print()
print("mean of the parts:", sum(parts) / len(parts))

T-3103 pulls `w` down (its part is positive). The other three pull it up, and T-3104, the longest ticket, pulls hardest. You use this table again in the next lesson.

## 4. The update rule: step downhill

The update rule is: new w = w − learning rate × gradient. The **learning rate** sets how far each change goes. One use of the rule is a **step**. Many steps are **gradient descent**.

Priya starts at `w = 2` with a learning rate of 0.05. The next cell makes her three steps. For each step, it draws a text trace: `●` is `w` after the step, and `|` is the best `w`, 6.27.

> **Predict.** The learning rate stays the same. Do the steps get longer, shorter or stay the same? Then run the cell.

In [ ]:
def trace(w, low=0, high=12, width=48, best=188 / 30):
    """One line: · for the w axis from low to high, | at the best w, ● at w."""
    line = ["·"] * (width + 1)
    line[round((best - low) / (high - low) * width)] = "|"
    if low <= w <= high:
        line[round((w - low) / (high - low) * width)] = "●"
    return "".join(line)

w = 2
rate = 0.05
rows = []
print(f"start   {trace(w)}  w = {w}")
for step in range(1, 4):
    g = gradient(w)
    move = -rate * g
    w = w + move
    rows.append([step, round(w - move, 4), g, round(move, 4), round(w, 4), round(loss(w), 4)])
    print(f"step {step}  {trace(w)}  w = {w:.4g}")
print()
table(["Step", "w before", "Gradient", "Move", "w after", "Loss after"], rows)

> **Check.** You should see `w` go 2, 5.2, 6.0, 6.2, and the loss 9.75, 1.75, 1.25. The moves are +3.2, +0.8 and +0.2: the steps get shorter by themselves, because the gradient shrinks near the bottom.

> **Your turn: change one thing.** In the cell above, change `range(1, 4)` to `range(1, 6)` and run it again. After five steps, `w` is 6.2625 and the loss is 1.2168: it rounds to 1.217, the lowest loss. Then change it back.

## 5. Two parameters: partial derivatives

Tomás adds a second parameter, a fixed number of minutes `b` for every ticket: minutes = w × length + b. The loss now depends on two knobs. Priya nudges **one knob at a time** and keeps the other one fixed.

> **Predict.** Does the loss go up or down when `w` grows a little? When `b` grows a little? Then run the cell.

In [ ]:
def loss_2(w, b):
    """Mean squared error of the model minutes = w × length + b."""
    squared = [(y - (w * x + b)) ** 2 for x, y in zip(lengths, minutes)]
    return sum(squared) / len(squared)

start = loss_2(6, 0)
rows = [["Start", 6, 0, round(start, 4), "", ""]]
for name, w, b, nudge in [("Nudge w by 0.1", 6.1, 0, 0.1), ("Nudge b by 0.1", 6, 0.1, 0.1)]:
    new = loss_2(w, b)
    rows.append([name, w, b, round(new, 4), round(new - start, 4), round((new - start) / nudge, 4)])
table(["Change", "w", "b", "Loss", "Change in loss", "Change ÷ nudge"], rows, digits=4)

tiny = 0.000001
print()
print("partial derivative for w:", round((loss_2(6 + tiny, 0) - start) / tiny, 3))
print("partial derivative for b:", round((loss_2(6, 0 + tiny) - start) / tiny, 3))

> **Check.** You should see a loss of 1.425 after the nudge of `w` and 1.61 after the nudge of `b`. With a very small nudge, the slopes are −4 and −1.5: the **partial derivatives**. The gradient is the list of both: (−4, −1.5). Both are negative, so the update rule makes both `w` and `b` a little larger.

## 6. Guided practice: the loss at w = 7

> **Your turn.** Replace each `...` with a calculation for `w = 7`. You can use a list comprehension, such as `[7 * x for x in lengths]`, or write the four numbers. Then run the check cell.

In [ ]:
predictions_7 = ...   # 7 × length, for each ticket
errors_7 = ...        # actual − prediction, for each ticket
squares_7 = ...       # each error squared
loss_7 = ...          # the mean of the squares
gradient_7 = ...      # 15w − 94 at w = 7
new_w_7 = ...         # one step from w = 7 with learning rate 0.05

Run the check cell. Each line says "Check passed." or what to look at again.

In [ ]:
expect("the list of predictions", predictions_7, [7, 14, 21, 28])
expect("the list of errors", errors_7, [0, -1, -4, -2])
expect("the list of squared errors", squares_7, [0, 1, 16, 4])
expect("the loss", loss_7, 5.25)
expect("the gradient", gradient_7, 11)
expect("the new w", new_w_7, 6.45)

One ticket, T-3103, gives 16 of the 21. The gradient is positive, so the update rule makes `w` smaller.

## 7. Independent variation: start on the other side

> **Your turn.** Start at `w = 8` with a learning rate of 0.05. Calculate the loss at `w = 8`. Make two steps with the update rule. Then calculate the loss after the second step. Run the check cell.

In [ ]:
loss_at_8 = ...
w_after_1 = ...
w_after_2 = ...
loss_after_2 = ...

In [ ]:
expect("the loss at w = 8", loss_at_8, 23.75)
expect("w after step 1", w_after_1, 6.7)
expect("w after step 2", w_after_2, 6.375, tol=0.001)
expect("the loss after step 2", loss_after_2, 1.30, tol=0.01)

Here the moves are negative, because the gradient is positive: the model predicts too many minutes, so `w` gets smaller. The update rule finds the direction by itself.

## 8. Failure case: plain errors cancel

A report says: "The model with `w = 6.3` has a mean error of 0 minutes. It is perfect."

> **Predict.** What are the four errors at `w = 6.3`? What is their mean, and what is the MSE? Then run the cell.

In [ ]:
w = 6.3
errors = [round(y - w * x, 2) for x, y in zip(lengths, minutes)]
print("errors:    ", errors)
print("mean error:", round(sum(errors) / len(errors), 6) + 0.0)
print("MSE:       ", round(loss(w), 4))

> **Check.** You should see the errors 0.7, 0.4, −1.9 and 0.8, a mean error of 0.0 and an MSE of 1.225. The errors cancel, although every prediction is wrong. **Fix:** square the errors (MSE), or take their absolute values (mean absolute error), before you take the mean.

In professional work, a library computes the loss for you: for example `mean_squared_error` in scikit-learn, or `MSELoss` in PyTorch. In PyTorch, `loss.backward()` computes the gradient for every parameter.

## Check your understanding and recap

Answer the **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/math/m04/loss-and-gradients/). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you calculated the loss of the ticket model: 1.75 at `w = 6`, and a lowest loss of 1.217 at `w ≈ 6.27`. The gradient, 15w − 94, is the slope of the loss: its sign gives the direction. The update rule, w − learning rate × gradient, took `w` from 2 to 6.2 in three steps. With two parameters, each one has a partial derivative.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Optimization behaviour](https://learning.nextia-ai.com/courses/math/m04/optimization-behaviour/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m04/loss-and-gradients/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.